# Parte 2 — Modelagem Preditiva

O objetivo desta etapa é desenvolver um modelo de regressão para prever o `klike_score`, pontuação de 0 a 100 que representa a qualidade geral de um criativo publicitário.

A modelagem foi orientada por um cenário de uso **anterior à veiculação da campanha**, no qual o anunciante deseja avaliar seu vídeo e identificar oportunidades de melhoria antes de publicá-lo.

Para isso, foram selecionadas apenas variáveis disponíveis nesse momento, evitando a utilização de métricas de performance observadas após a veiculação.

A abordagem prioriza modelos simples, avaliação de desempenho e interpretabilidade, considerando uma possível aplicação dos resultados no sistema de recomendações.

In [1]:
# Manipulação de dados
import numpy as np
import pandas as pd

# Pré-processamento
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, KFold, cross_val_score

# Modelos
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

# Avaliação
from sklearn.metrics import (mean_absolute_error, root_mean_squared_error, r2_score)

## 1. Seleção de features e divisão dos dados

Foram selecionadas 13 variáveis preditoras, divididas em dois grupos:

- **Contexto da campanha:** plataforma, categoria, objetivo, faixa etária do público-alvo e retargeting.
- **Características do criativo:** duração, formato, presença de legendas, CTA, hook, rosto humano, densidade de texto e proporção entre música e voz.

Métricas como CTR, ROAS, conversões e engajamento foram excluídas, pois não estariam disponíveis antes da veiculação. O identificador da campanha e a data de veiculação também não foram utilizados como preditores.

O dataset foi dividido em 80% para treinamento (400 campanhas) e 20% para teste (100 campanhas), utilizando uma divisão aleatória com `random_state=26` para garantir reprodutibilidade.

O conjunto de teste foi mantido separado do ajuste dos modelos e utilizado para avaliar seu desempenho em campanhas não vistas durante o treinamento.

In [2]:
df_modelo = pd.read_csv('../data/klike_challenge_dataset.csv')
df_modelo.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 26 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   campaign_id          500 non-null    str    
 1   date                 500 non-null    str    
 2   platform             500 non-null    str    
 3   category             500 non-null    str    
 4   objective            500 non-null    str    
 5   target_audience_age  500 non-null    str    
 6   is_retargeting       500 non-null    bool   
 7   video_duration_s     461 non-null    float64
 8   format               500 non-null    str    
 9   has_subtitle         454 non-null    object 
 10  has_cta              500 non-null    bool   
 11  has_hook             500 non-null    bool   
 12  has_face             500 non-null    bool   
 13  text_density         500 non-null    str    
 14  music_voice_ratio    462 non-null    float64
 15  impressions          500 non-null    int64  
 16  c

In [3]:
X = df_modelo[['platform', 'category', 'objective', 'target_audience_age', 'format', 'text_density', 'video_duration_s', 'music_voice_ratio', 
               'is_retargeting', 'has_subtitle', 'has_cta', 'has_hook', 'has_face']]
y = df_modelo['klike_score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=26)
print(f'Treino: {X_train.shape[0]} campanhas')
print(f'Teste: {y_test.shape[0]} campanhas')


Treino: 400 campanhas
Teste: 100 campanhas


## 2. Pré-processamento

As variáveis foram organizadas por tipo para receber transformações adequadas:

- **Numéricas:** imputação pela média e padronização com `StandardScaler`. A média foi escolhida para `music_voice_ratio` por sua distribuição aproximadamente simétrica. Para `video_duration_s`, foi mantida como estratégia simples de preenchimento, embora a mediana pudesse ser mais robusta diante de assimetria ou valores extremos. A padronização foi aplicada para evitar diferenças de escala no treinamento do Ridge.
- **Categóricas nominais:** imputação pela categoria mais frequente e aplicação de `OneHotEncoder`.
- **Categóricas ordinais:** imputação pela categoria mais frequente e codificação ordenada com `OrdinalEncoder`.
- **Booleanas:** imputação pelo valor mais frequente, quando necessário, preservando sua representação binária.

As transformações foram organizadas com `Pipeline` e `ColumnTransformer`, permitindo aplicar diferentes tratamentos aos grupos de variáveis.

O pré-processamento foi ajustado apenas com os dados de treinamento e posteriormente aplicado aos dados de teste, evitando vazamento de informações.

Após as transformações, as 13 features originais passaram a ser representadas por 25 colunas numéricas.

In [4]:
numericos = ['video_duration_s', 'music_voice_ratio']
categoricas = ['platform', 'category', 'objective', 'format']
ordinais = ['text_density', 'target_audience_age']
booleanos = ['is_retargeting', 'has_subtitle', 'has_cta', 'has_hook', 'has_face']

In [5]:
numerico_pipeline = Pipeline([("imputer", SimpleImputer(strategy="mean")), ("scaler", StandardScaler())])
categorica_pipeline = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", OneHotEncoder(handle_unknown="ignore"))])
ordinal_pipeline = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", OrdinalEncoder(
    categories=[["low", "medium", "high"], ["18-24", "25-34", "35-44", "45+"]], handle_unknown="error"))])
booleano_pipeline = Pipeline([("imputer", SimpleImputer(strategy="most_frequent"))])

In [6]:
preprocessor = ColumnTransformer([
    ("numeric", numerico_pipeline, numericos),
    ("categorical", categorica_pipeline, categoricas),
    ("ordinal", ordinal_pipeline, ordinais),
    ("boolean", booleano_pipeline, booleanos)])

In [7]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

## 3. Treinamento e avaliação dos modelos

Foram avaliadas duas abordagens de regressão:

- **Ridge Regression:** modelo linear com regularização L2, escolhido pela simplicidade e interpretabilidade.
- **Random Forest Regressor:** modelo baseado em múltiplas árvores de decisão, capaz de capturar relações não lineares e interações entre variáveis.

Também foi utilizado um `DummyRegressor` como baseline, prevendo a média do `klike_score` no conjunto de treinamento.

O desempenho foi avaliado por três métricas:

- **MAE:** erro absoluto médio, expresso em pontos do score.
- **RMSE:** erro que penaliza mais intensamente previsões com desvios elevados.
- **R²:** desempenho do modelo em relação à previsão constante da média, considerando a variabilidade do alvo.

A comparação com o baseline permite verificar se os modelos conseguem extrair informações preditivas das features selecionadas.

In [8]:
ridge = Ridge()
ridge.fit(X_train_processed, y_train)

y_pred_ridge = ridge.predict(X_test_processed)

In [9]:
mae_ridge = mean_absolute_error(y_test, y_pred_ridge)
rmse_ridge = root_mean_squared_error(y_test, y_pred_ridge)
r2_ridge = r2_score(y_test, y_pred_ridge)

print(f"MAE: {mae_ridge:.2f}")
print(f"RMSE: {rmse_ridge:.2f}")
print(f"R²: {r2_ridge:.3f}")

MAE: 8.59
RMSE: 10.39
R²: 0.617


In [10]:
dummy = DummyRegressor()
dummy.fit(X_train, y_train)

y_pred_dummy = dummy.predict(X_test)

In [11]:
mae_dummy = mean_absolute_error(y_test, y_pred_dummy)
rmse_dummy = root_mean_squared_error(y_test, y_pred_dummy)
r2_dummy = r2_score(y_test, y_pred_dummy)

print(f"MAE: {mae_dummy:.2f}")
print(f"RMSE: {rmse_dummy:.2f}")
print(f"R²: {r2_dummy:.3f}")

MAE: 13.84
RMSE: 16.80
R²: -0.001


In [12]:
rf = RandomForestRegressor(random_state=26)
rf.fit(X_train_processed, y_train)

y_pred_rf = rf.predict(X_test_processed)

In [13]:
mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = root_mean_squared_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print(f"MAE: {mae_rf:.2f}")
print(f"RMSE: {rmse_rf:.2f}")
print(f"R²: {r2_rf:.3f}")

MAE: 8.31
RMSE: 10.43
R²: 0.614


### Comparação inicial

| Modelo | MAE | RMSE | R² |
|---|---:|---:|---:|
| DummyRegressor | 13,84 | 16,80 | -0,001 |
| Ridge | 8,59 | 10,39 | 0,617 |
| Random Forest | 8,31 | 10,43 | 0,614 |

Os dois modelos de regressão superaram o baseline, indicando que as features selecionadas contêm informações úteis para prever o `klike_score`.

O Random Forest apresentou o menor MAE, enquanto o Ridge obteve resultados ligeiramente melhores em RMSE e R². Como as diferenças foram pequenas, a comparação também considerou o comportamento dos modelos nos dados de treinamento e na validação cruzada.

## 4. Análise de overfitting

Para investigar a capacidade de generalização dos modelos, seus resultados nos conjuntos de treinamento e teste foram comparados.

Uma diferença elevada entre o desempenho nesses conjuntos pode indicar overfitting, situação na qual o modelo se ajusta excessivamente aos dados de treinamento e apresenta dificuldade para generalizar para novas observações.

In [14]:
y_train_pred_ridge = ridge.predict(X_train_processed)
y_train_pred_rf = rf.predict(X_train_processed)

In [15]:
mae_rf_train = mean_absolute_error(y_train, y_train_pred_rf)
rmse_rf_train = root_mean_squared_error(y_train, y_train_pred_rf)
r2_rf_train = r2_score(y_train, y_train_pred_rf)

mae_ridge_train = mean_absolute_error(y_train, y_train_pred_ridge)
rmse_ridge_train = root_mean_squared_error(y_train, y_train_pred_ridge)
r2_ridge_train = r2_score(y_train, y_train_pred_ridge)

overfitting_df = pd.DataFrame({"Modelo": ["Ridge", "Random Forest"], "MAE Treino": [mae_ridge_train, mae_rf_train], "MAE Teste": [mae_ridge, mae_rf],
    "RMSE Treino": [rmse_ridge_train, rmse_rf_train], "RMSE Teste": [rmse_ridge, rmse_rf], "R² Treino": [r2_ridge_train, r2_rf_train], "R² Teste": [r2_ridge, r2_rf]})
display(overfitting_df.round(3))

,Modelo,MAE Treino,MAE Teste,RMSE Treino,RMSE Teste,R² Treino,R² Teste
0,Ridge,7.614,8.588,9.569,10.393,0.606,0.617
1,Random Forest,2.926,8.314,3.750,10.432,0.940,0.614


O Random Forest apresentou R² de 0,940 no treinamento, mas 0,614 no teste. Seu MAE aumentou de 2,93 para 8,31, evidenciando um ajuste excessivo aos dados de treinamento.

O Ridge apresentou resultados mais próximos entre treino e teste, com MAE de 7,61 e 8,59, respectivamente, sugerindo maior capacidade de generalização nessa divisão dos dados.

Embora o Random Forest tenha alcançado um MAE ligeiramente menor no teste, sua vantagem preditiva foi pequena diante da diferença observada entre treinamento e teste.

## 5. Validação cruzada

Para complementar a avaliação baseada em uma única divisão dos dados, foi realizada validação cruzada com cinco folds sobre o conjunto de treinamento.

Em cada iteração, quatro partes foram utilizadas para ajuste e uma para validação, utilizando o MAE como métrica de comparação.

O pré-processamento foi incorporado aos pipelines dos modelos para garantir que imputação, codificação e padronização fossem ajustadas apenas com os dados de treinamento de cada fold.

In [16]:
ridge_cv = Pipeline([("preprocessor", preprocessor), ("model", Ridge())])
rf_cv = Pipeline([("preprocessor", preprocessor), ("model", RandomForestRegressor(random_state=26))])

In [17]:
cv = KFold(n_splits=5, shuffle=True, random_state=26)

scores_ridge = cross_val_score(ridge_cv, X_train, y_train, cv=cv, scoring="neg_mean_absolute_error")
scores_rf = cross_val_score(rf_cv, X_train, y_train, cv=cv, scoring="neg_mean_absolute_error")

In [18]:
mae_cv_ridge = -scores_ridge
mae_cv_rf = -scores_rf

cv_resultados = pd.DataFrame({"Modelo": ["Ridge", "Random Forest"], "MAE médio": [mae_cv_ridge.mean(), mae_cv_rf.mean()], "Desvio-padrão": [mae_cv_ridge.std(), mae_cv_rf.std()]})
display(cv_resultados.round(2))

,Modelo,MAE médio,Desvio-padrão
0,Ridge,8.06,0.50
1,Random Forest,7.80,0.53


### Resultados da validação cruzada

| Modelo | MAE médio | Desvio-padrão |
|---|---:|---:|
| Ridge | 8,06 | 0,50 |
| Random Forest | 7,80 | 0,53 |

Embora o Random Forest tenha apresentado MAE ligeiramente inferior no teste e na validação cruzada, sua vantagem foi pequena e acompanhada de uma diferença expressiva entre o desempenho no treinamento e no teste, indicando overfitting.

Por isso, optei pelo Ridge como modelo final. Além de apresentar desempenho preditivo competitivo, ele demonstrou maior consistência entre treino e teste e permite interpretar diretamente os coeficientes das variáveis, facilitando a análise dos fatores associados ao `klike_score`.

## 6. Interpretação das features

Para investigar a contribuição das variáveis para as previsões, foram analisados os coeficientes do Ridge.

O sinal dos coeficientes indica a direção da associação com o score previsto, mantendo as demais variáveis constantes. Já sua magnitude depende da escala e da codificação de cada feature.

Os coeficientes foram ordenados por valor absoluto para facilitar a identificação das associações mais expressivas.

In [19]:
feature_names = preprocessor.get_feature_names_out()

coef_df = pd.DataFrame({"feature": feature_names, "coefficient": ridge.coef_})
coef_df = coef_df.sort_values(by="coefficient", key=abs, ascending=False)

display(coef_df)

,feature,coefficient
23,boolean__has_hook,16.882692
24,boolean__has_face,10.415634
22,boolean__has_cta,8.062488
21,boolean__has_subtitle,7.482633
20,boolean__is_retargeting,4.901445
2,categorical__platform_LinkedIn,-3.150059
4,categorical__platform_TikTok,2.972062
5,categorical__category_App Install,-2.508804
18,ordinal__text_density,-2.487101
9,categorical__category_SaaS,2.068826


### Principais resultados

Os maiores coeficientes positivos foram encontrados para:

- **Hook:** +16,88
- **Rosto humano:** +10,42
- **CTA:** +8,06
- **Legendas:** +7,48

Como essas variáveis são booleanas, seus coeficientes representam a diferença no score previsto entre a ausência e a presença do atributo, mantendo as demais características constantes.

Esses resultados são coerentes com parte dos achados da EDA: hooks e rostos humanos apresentaram associações positivas com CTR e engajamento, enquanto CTAs estiveram associados a melhores métricas de conversão e retorno financeiro.

A densidade de texto apresentou coeficiente negativo (-2,49), também coerente com os padrões identificados na análise exploratória.

Entretanto, os coeficientes não representam efeitos causais. Além disso, diferenças de escala e codificação limitam a comparação direta entre todas as variáveis. As interpretações devem ser entendidas como associações aprendidas pelo modelo, não como garantias de melhoria do score ou da performance.

## 7. Conclusões e limitações

A modelagem demonstrou que características dos criativos e informações de configuração das campanhas permitem prever parte da variação do `klike_score`, mesmo sem utilizar métricas de performance pós-veiculação.

O Ridge foi selecionado como modelo final, apresentando MAE de 8,59 pontos e R² de 0,617 no conjunto de teste. Seu desempenho superou significativamente o baseline, com resultados competitivos em relação ao Random Forest e menor evidência de overfitting.

A interpretação dos coeficientes destacou atributos como hook, rosto humano, CTA e legendas, estabelecendo conexões com os padrões observados na análise exploratória.

**Limitações:** o dataset possui apenas 500 campanhas e não informa como o `klike_score` foi originalmente calculado. Dessa forma, não é possível determinar quais fatores efetivamente compõem a pontuação nem interpretar as associações aprendidas como relações causais.

Os resultados oferecem uma base para a construção do Recommendations Engine, que utilizará os padrões identificados para formular recomendações contextualizadas, quantificadas e priorizadas, sem apresentar melhorias estimadas como resultados garantidos.